# GEPA Prompt Optimization

This notebook optimizes the base agent's system prompt using GEPA, scored by the aligned judge.

GEPA runs 5 independent optimization passes on disjoint 20-question subsets drawn from a
100-question synthetic pool, each with a budget of 100 scorer calls. Results are checkpointed
to a Delta table after every run for resilience against rate limits and interruptions.

The best prompt is registered to the Prompt Registry and promoted to `@production`. Downstream
notebooks consume this prompt:
- `07-AgentSkillsGeneration.ipynb` generates skills using the optimized prompt
- `08_create_agent_with_skills.ipynb` builds the skills-enhanced agent
- `09-Evaluation.ipynb` evaluates both agents on a held-out dataset

**Prerequisites:**
- `03_create_agent_definition.ipynb` has written `agent.py`
- `04-Evaluation.ipynb` has created the evaluation dataset
- `05-JudgeAlignment.ipynb` has produced the aligned judge

In [ ]:
%pip install -U -qqqq backoff databricks-openai uv databricks-agents "mlflow>=3.9" dspy databricks-mcp langgraph-checkpoint-postgres "psycopg[binary,pool]" databricks-langchain langgraph
dbutils.library.restartPython()

In [ ]:
import json
import os
import random
import re
import importlib
import hashlib
import warnings
import logging
from contextlib import contextmanager
from pathlib import Path

import mlflow

CONFIG = json.loads(Path("config/atbat_assistant.json").read_text())

EXPERIMENT_ID = CONFIG["mlflow"]["experiment_id"]
PROMPT_NAME = CONFIG["prompt_registry"]["prompt_name"]
REFLECTION_MODEL = CONFIG["prompt_registry"]["reflection_model"]
JUDGE_MODEL = CONFIG["llm"]["judge_model"]
ALIGNED_JUDGE_NAME = CONFIG["judges"]["aligned_judge_name"]

N_RUNS = 5
POOL_SIZE = 100
SUBSET_SIZE = 20
MAX_METRIC_CALLS = 100
RANDOM_SEED = 42

JUDGE_EXPERIMENT_ID = EXPERIMENT_ID

_scope = CONFIG["prompt_registry_auth"]["secret_scope_name"]
_sp_id = dbutils.secrets.get(scope=_scope, key=CONFIG["prompt_registry_auth"]["oauth_client_id_key"])
_sp_secret = dbutils.secrets.get(scope=_scope, key=CONFIG["prompt_registry_auth"]["oauth_client_secret_key"])

@contextmanager
def _sp_auth():
    """Temporarily swap to Service Principal OAuth for prompt registry operations."""
    saved_token = os.environ.pop("DATABRICKS_TOKEN", None)
    os.environ["DATABRICKS_CLIENT_ID"] = _sp_id
    os.environ["DATABRICKS_CLIENT_SECRET"] = _sp_secret
    try:
        yield
    finally:
        os.environ.pop("DATABRICKS_CLIENT_ID", None)
        os.environ.pop("DATABRICKS_CLIENT_SECRET", None)
        if saved_token is not None:
            os.environ["DATABRICKS_TOKEN"] = saved_token

print("SP auth context manager ready.")

with _sp_auth():
    parent_experiment = mlflow.get_experiment(EXPERIMENT_ID)
    experiment_06 = mlflow.set_experiment(f"{parent_experiment.name}-06-prompt-optimization")
EXPERIMENT_ID = experiment_06.experiment_id
print(f"Using dedicated experiment: {experiment_06.name} (ID: {EXPERIMENT_ID})")
print(f"Aligned judge will be loaded from original experiment: {JUDGE_EXPERIMENT_ID}")

print(f"Experiment config: {N_RUNS} GEPA runs, {SUBSET_SIZE} examples per run, {MAX_METRIC_CALLS} scorer calls per run")
print(f"Total scorer calls budget: ~{N_RUNS * MAX_METRIC_CALLS}")
print("Best prompt will be registered to @production for downstream notebooks (07, 08, 09).")

## Step 1: Load or Generate Question Pool

If the pool has been generated before and saved to a Delta table, load it. Otherwise,
generate a pool of 100 diverse questions via FMAPI and persist it for future runs.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

POOL_TABLE = f"{CONFIG['workspace']['catalog']}.{CONFIG['workspace']['schema']}.gepa_optimization_pool"

_pool_loaded = False
pool = []

if spark.catalog.tableExists(POOL_TABLE):
    _pool_df = spark.table(POOL_TABLE)
    _pool_count = _pool_df.count()
    if _pool_count >= POOL_SIZE:
        print(f"Loading existing pool from {POOL_TABLE} ({_pool_count} rows)")
        pool = [json.loads(row["example_json"]) for row in _pool_df.orderBy("idx").collect()]
        pool = pool[:POOL_SIZE]
        _pool_loaded = True
    else:
        print(f"Pool table exists but only has {_pool_count} rows (need {POOL_SIZE}). Regenerating.")

if not _pool_loaded:
    from openai import OpenAI
    from databricks.sdk import WorkspaceClient

    w = WorkspaceClient()
    fmapi_client: OpenAI = w.serving_endpoints.get_open_ai_client()

    OPTIMIZATION_GEN_MODEL = CONFIG["llm"]["judge_model"].replace("databricks:/", "")
    print(f"Generating {POOL_SIZE}-question pool with model: {OPTIMIZATION_GEN_MODEL}")

    GENERATION_PROMPT = f"""You are an expert at generating realistic user questions for a baseball hitting analysis AI assistant.

The assistant helps batters prepare for matchups against specific pitchers. It can look up player IDs, get pitcher tendencies by count/hand/runners, pull historical batter-pitcher matchups, look up arsenals, find similar players via embeddings, recommend lineups, and query a Genie Space for SQL-based analytics.

Generate exactly {POOL_SIZE} training examples as a JSON array. Each example should be an object with exactly two keys:
- "inputs": a realistic user question (single string)
- "expected_response": a 1-2 sentence description of what a good response should contain

Write questions the way a real hitting coach, analyst, or front office person would ask them -- natural, varied in complexity, and covering a range of scenarios:
- Quick factual lookups ("What does Gerrit Cole throw?")
- Matchup scouting ("How will Max Fried attack George Springer?")
- Situational strategy ("How does Corbin Burnes change with runners in scoring position?")
- Lineup construction ("Who should I start against Logan Webb?")
- Comparative analysis ("How does Fried's slider compare to other lefty sliders?")
- Open-ended at-bat previews ("What should Freddie Freeman expect facing Zac Gallen?")
- Analytics questions ("What's the average spin rate on NYY fastballs this year?")

Use real MLB player names and teams. Use 3-letter abbreviations: TEX,CHC,LAA,LAD,STL,PHI,ARI,OAK,TBR,MIN,CLE,CHW,NYM,COL,SEA,MIA,SDP,WSN,HOU,SFG,CIN,BAL,KCR,PIT,ATL,NYY,DET,MIL,TOR,BOS,ATH

Only reference 2024 & 2025 seasons. Keep "expected_response" to 1-2 sentences.
Ensure all {POOL_SIZE} questions are distinct and cover a broad range of the scenarios above.

Return ONLY a valid JSON array, no other text."""

    response = fmapi_client.chat.completions.create(
        model=OPTIMIZATION_GEN_MODEL,
        messages=[{"role": "user", "content": GENERATION_PROMPT}],
        temperature=0.9,
        max_tokens=30000,
    )

    raw_output = response.choices[0].message.content
    print(f"Raw output length: {len(raw_output)} chars")
    print(f"First 500 chars:\n{raw_output[:500]}")

In [ ]:
if not _pool_loaded:
    cleaned = re.sub(r'^```(?:json)?\s*', '', raw_output.strip())
    cleaned = re.sub(r'\s*```$', '', cleaned)

    json_match = re.search(r'\[.*\]', cleaned, re.DOTALL)
    if json_match:
        generated_examples = json.loads(json_match.group())
    else:
        generated_examples = json.loads(cleaned)

    print(f"Parsed {len(generated_examples)} examples from FMAPI")

    if len(generated_examples) < POOL_SIZE:
        print(f"WARNING: Got {len(generated_examples)} instead of {POOL_SIZE}. Using what we have.")
        actual_pool_size = len(generated_examples)
        actual_subset_size = actual_pool_size // N_RUNS
        print(f"Adjusted: {N_RUNS} subsets of {actual_subset_size} each")
    else:
        generated_examples = generated_examples[:POOL_SIZE]
        actual_subset_size = SUBSET_SIZE

    for ex in generated_examples:
        raw_expectation = ex.get("expected_response") or ex.get("expectations", "")
        if isinstance(raw_expectation, dict):
            expectations_dict = raw_expectation
        else:
            expectations_dict = {"expected_response": str(raw_expectation)}

        pool.append({
            "inputs": {
                "input": [{"role": "user", "content": ex["inputs"]}]
            },
            "expectations": expectations_dict,
        })

    random.seed(RANDOM_SEED)
    random.shuffle(pool)

    _pool_schema = StructType([
        StructField("idx", IntegerType()),
        StructField("example_json", StringType()),
    ])
    _pool_rows = [{"idx": i, "example_json": json.dumps(p)} for i, p in enumerate(pool)]
    spark.createDataFrame(_pool_rows, schema=_pool_schema).write.mode("overwrite").saveAsTable(POOL_TABLE)
    print(f"Saved {len(pool)} examples to {POOL_TABLE}")
else:
    actual_subset_size = SUBSET_SIZE

subsets = [pool[i * actual_subset_size:(i + 1) * actual_subset_size] for i in range(N_RUNS)]

print(f"\nPool: {len(pool)} examples, {len(subsets)} disjoint subsets of {actual_subset_size} each")
for i, s in enumerate(subsets):
    q = s[0]["inputs"]["input"][0]["content"]
    print(f"  Subset {i}: '{q[:80]}...'")

assert isinstance(pool[0]["expectations"], dict), "expectations must be a dict!"
print("\nStructure validation passed")

## Step 2: Load Aligned Judge

In [ ]:
from mlflow.genai.scorers import get_scorer, Scorer
from mlflow.entities import Feedback
import re as _re
import sys as _sys
import litellm as _litellm

GUARDRAIL_SENTINEL = -1.0
_GUARDRAIL_RE = _re.compile(r'(?<![_a-zA-Z])arsenal(?![_a-zA-Z])', _re.IGNORECASE)


def _scrub(text):
    """Replace guardrail-triggering baseball terms in a string."""
    if isinstance(text, str):
        return _GUARDRAIL_RE.sub('pitch repertoire', text)
    return text


def _scrub_messages(messages):
    if not messages or not isinstance(messages, list):
        return
    for msg in messages:
        if not isinstance(msg, dict):
            continue
        c = msg.get("content")
        if isinstance(c, str):
            msg["content"] = _scrub(c)
        elif isinstance(c, list):
            for part in c:
                if isinstance(part, dict) and isinstance(part.get("text"), str):
                    part["text"] = _scrub(part["text"])


_original_completion = _litellm.completion

def _sanitized_completion(*args, **kwargs):
    _scrub_messages(kwargs.get("messages"))
    if args:
        for a in args:
            if isinstance(a, list):
                _scrub_messages(a)
    return _original_completion(*args, **kwargs)

_litellm.completion = _sanitized_completion

_patched_count = 0
for _mod_name in list(_sys.modules.keys()):
    _mod = _sys.modules.get(_mod_name)
    if _mod is None:
        continue
    for _attr in ['completion', 'litellm_completion']:
        try:
            _ref = getattr(_mod, _attr, None)
            if _ref is _original_completion:
                setattr(_mod, _attr, _sanitized_completion)
                _patched_count += 1
        except Exception:
            pass

print(f"Patched litellm.completion + {_patched_count} cached refs across loaded modules")

_inner_judge = get_scorer(name=ALIGNED_JUDGE_NAME, experiment_id=JUDGE_EXPERIMENT_ID)
print(f"Loaded aligned judge: {_inner_judge.name}")

_patched_count2 = 0
for _mod_name in list(_sys.modules.keys()):
    _mod = _sys.modules.get(_mod_name)
    if _mod is None:
        continue
    for _attr in ['completion', 'litellm_completion']:
        try:
            _ref = getattr(_mod, _attr, None)
            if _ref is _original_completion:
                setattr(_mod, _attr, _sanitized_completion)
                _patched_count2 += 1
        except Exception:
            pass
if _patched_count2:
    print(f"Patched {_patched_count2} additional refs found after scorer load")


class GuardrailSafeScorer(Scorer):
    """Wraps an existing scorer and catches AI Gateway guardrail errors.

    Returns Feedback with value=GUARDRAIL_SENTINEL (-1.0) so the objective
    function can detect and exclude these from the running average.
    Also scrubs inputs/outputs before delegating and detects predict_fn fallbacks.
    """
    name: str = ALIGNED_JUDGE_NAME
    _delegate: object = None

    class Config:
        underscore_attrs_are_private = True

    def __init__(self, delegate, **kwargs):
        super().__init__(**kwargs)
        self._delegate = delegate

    def __call__(self, *, inputs=None, outputs=None, expectations=None, trace=None):
        if isinstance(outputs, str) and "(Skipped: input guardrail triggered" in outputs:
            logging.warning("predict_fn returned guardrail fallback, marking as sentinel")
            return Feedback(
                name=self.name,
                value=GUARDRAIL_SENTINEL,
                rationale="GUARDRAIL_SKIP",
            )
        scrubbed_outputs = _scrub(outputs) if isinstance(outputs, str) else outputs
        try:
            return self._delegate(
                inputs=inputs, outputs=scrubbed_outputs,
                expectations=expectations, trace=trace,
            )
        except Exception as e:
            if "guardrail" in str(e).lower() or "input_guardrail_triggered" in str(e):
                logging.warning(f"Guardrail triggered during scoring: {str(e)[:120]}")
                return Feedback(
                    name=self.name,
                    value=GUARDRAIL_SENTINEL,
                    rationale="GUARDRAIL_SKIP",
                )
            raise


aligned_judge = GuardrailSafeScorer(delegate=_inner_judge)
print(f"Wrapped judge with guardrail-safe handler (sentinel={GUARDRAIL_SENTINEL})")

## Step 3: Define Shared Helpers

The `predict_fn_factory` creates a predict function bound to a specific agent module,
letting us switch between `agent` (base) and `agent_with_skills` without code duplication.

In [ ]:
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=UserWarning, module='pydantic')
logging.getLogger('mlflow.genai.judges.instructions_judge').setLevel(logging.ERROR)
logging.getLogger('mlflow.tracing.fluent').setLevel(logging.ERROR)
logging.getLogger('mlflow.tracing.export.mlflow_v3').setLevel(logging.ERROR)
logging.getLogger('mlflow.tracing.provider').setLevel(logging.ERROR)


def _get(item, key, default=""):
    if isinstance(item, dict):
        return item.get(key, default)
    return getattr(item, key, default)


def _extract_compact_response(result) -> str:
    tool_calls = []
    final_text = ""

    for item in result.output:
        item_type = _get(item, "type")

        if item_type == "function_call":
            name = _get(item, "name", "unknown")
            args_str = _get(item, "arguments", "{}")
            if len(args_str) > 300:
                args_str = args_str[:300] + "..."
            tool_calls.append(f"  - {name}({args_str})")

        elif item_type == "message":
            content = _get(item, "content", [])
            if isinstance(content, list):
                for block in content:
                    block_type = _get(block, "type") if isinstance(block, dict) else getattr(block, "type", "")
                    if block_type == "output_text":
                        text = _get(block, "text", "") if isinstance(block, dict) else getattr(block, "text", "")
                        if text:
                            final_text = text
            elif isinstance(content, str):
                final_text = content

        elif item_type == "text":
            text = _get(item, "text", "")
            if text:
                final_text = text

    parts = []
    if tool_calls:
        parts.append("[Tool Calls]\n" + "\n".join(tool_calls))
    if final_text:
        parts.append("[Agent Analysis]\n" + final_text)

    compact = "\n\n".join(parts) if parts else "(no response)"
    return _scrub(compact)


def make_objective_function(judge_name, subset_size):
    """Create an objective function scoped to a specific run's subset size.

    Guardrail-skipped rows (sentinel = -1.0 / rationale = GUARDRAIL_SKIP)
    are excluded from the running average. For those rows we return the
    current mean of real scores so the candidate's aggregate is unaffected.
    """
    counter = {"count": 0, "scores": [], "skipped": 0, "total_seen": 0}

    def objective_function(scores: dict) -> float:
        feedback = scores.get(judge_name)
        counter["total_seen"] += 1

        if feedback and hasattr(feedback, 'feedback') and hasattr(feedback.feedback, 'value'):
            try:
                raw_score = float(feedback.feedback.value)
            except (ValueError, TypeError):
                raw_score = None
        else:
            raw_score = None

        is_sentinel = raw_score is not None and raw_score == GUARDRAIL_SENTINEL
        is_skip = (
            is_sentinel
            or (feedback and hasattr(feedback, 'feedback')
                and getattr(feedback.feedback, 'rationale', '') == "GUARDRAIL_SKIP")
        )

        if is_skip:
            counter["skipped"] += 1
            if counter["scores"]:
                current_mean = sum(counter["scores"]) / len(counter["scores"])
                return current_mean
            return float("nan")

        if raw_score is not None:
            normalized = raw_score / 5.0
            counter["scores"].append(normalized)

            if counter["total_seen"] == subset_size:
                n_real = len(counter["scores"])
                avg = sum(counter["scores"]) / n_real if n_real else 0.0
                counter["count"] += 1
                skipped = counter["skipped"]
                print(f"    Candidate #{counter['count']} avg: {avg:.4f} (0-1 scale) "
                      f"({n_real} scored, {skipped} guardrail-skipped)")
                counter["scores"] = []
                counter["skipped"] = 0
                counter["total_seen"] = 0

            return normalized

        return float("nan")

    return objective_function


def predict_fn_factory(agent_module_name: str, prompt_uri=None):
    """Return a predict_fn bound to the given agent module and prompt URI.

    If prompt_uri is None, defaults to the seed prompt (v1). Pass a specific
    URI so GEPA's load_prompt interception matches the URI in prompt_uris.
    """
    with _sp_auth():
        mod = importlib.import_module(agent_module_name)
        agent_instance = mod.AGENT
    last_hash = {"hash": None}
    _prompt_uri = prompt_uri or seed_prompt_ref.uri

    def predict_fn(input):
        with _sp_auth():
            prompt = mlflow.genai.load_prompt(_prompt_uri)
        system_content = prompt.format()

        prompt_hash = hashlib.md5(system_content.encode()).hexdigest()
        if prompt_hash != last_hash["hash"]:
            print(f"    New prompt candidate (hash: {prompt_hash[:8]})")
            last_hash["hash"] = prompt_hash

        if isinstance(input, dict) and "input" in input:
            user_message = input["input"][0]["content"]
        elif isinstance(input, list):
            user_message = input[0]["content"]
        else:
            user_message = str(input)

        messages = [
            {"role": "system", "content": system_content},
            {"role": "user", "content": user_message},
        ]

        mlflow.tracing.disable()
        try:
            result = agent_instance.predict({"input": messages})
        except Exception as e:
            mlflow.tracing.enable()
            err_str = str(e)
            if "input_guardrail_triggered" in err_str or "guardrail" in err_str.lower():
                logging.warning(f"Guardrail false positive, returning fallback: {user_message[:80]}")
                return "[Agent Analysis]\n(Skipped: input guardrail triggered on benign baseball query)"
            raise
        finally:
            mlflow.tracing.enable()

        return _extract_compact_response(result)

    return predict_fn


SEED_PROMPT_VERSION = 1
with _sp_auth():
    seed_prompt_ref = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}/{SEED_PROMPT_VERSION}")
print(f"Seed prompt (pinned to v{SEED_PROMPT_VERSION}): {seed_prompt_ref.uri}")
print("GEPA optimizes from this prompt. Best result is registered to @production.")
print("Helpers defined.")

## Step 4: Sanity Check

Run a single prediction with the base agent to verify extraction works before the full experiment.

In [ ]:
test_input = subsets[0][0]["inputs"]
q_text = test_input["input"][0]["content"]
print(f"Test question: {q_text[:150]}")

print("\n--- Testing agent (base) ---")
pfn = predict_fn_factory("agent")
output = pfn(test_input)
has_tools = "[Tool Calls]" in output
has_analysis = "[Agent Analysis]" in output
print(f"  Output length: {len(output)} chars")
print(f"  Has [Tool Calls]: {has_tools}")
print(f"  Has [Agent Analysis]: {has_analysis}")
if not has_analysis:
    print(f"  WARNING: No [Agent Analysis] found. Check agent output format.")
else:
    print(f"  OK")

## Step 5: Run GEPA Optimization (with checkpoint/resume)

5 GEPA runs on the base agent (`agent.py`), starting from the original prompt (v1).
Each run uses a disjoint 20-question subset and 100 scorer calls.

Results are checkpointed to a Delta table after every run. If the notebook is interrupted
(e.g., rate limits), re-running this cell resumes from where it left off.

In [ ]:
from mlflow.genai.optimize import GepaPromptOptimizer
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
import time

CHECKPOINT_TABLE = f"{CONFIG['workspace']['catalog']}.{CONFIG['workspace']['schema']}.gepa_experiment_checkpoint"

_checkpoint_schema = StructType([
    StructField("agent_type", StringType()),
    StructField("run_idx", IntegerType()),
    StructField("initial_score", DoubleType()),
    StructField("final_score", DoubleType()),
    StructField("prompt_template", StringType()),
    StructField("elapsed_seconds", DoubleType()),
])

if spark.catalog.tableExists(CHECKPOINT_TABLE):
    _checkpoint_df = spark.table(CHECKPOINT_TABLE)
    results = [row.asDict() for row in _checkpoint_df.collect()]
    print(f"Resumed: {len(results)} completed runs loaded from {CHECKPOINT_TABLE}")
else:
    results = []
    print(f"Starting fresh experiment (no table {CHECKPOINT_TABLE})")


def _save_checkpoint():
    df = spark.createDataFrame(results, schema=_checkpoint_schema)
    df.write.mode("overwrite").saveAsTable(CHECKPOINT_TABLE)
    print(f"  Checkpoint saved to {CHECKPOINT_TABLE} ({len(results)} runs)")


def _is_done(agent_type, run_idx):
    return any(r["agent_type"] == agent_type and r["run_idx"] == run_idx for r in results)


def _get_result(agent_type, run_idx):
    for r in results:
        if r["agent_type"] == agent_type and r["run_idx"] == run_idx:
            return r
    return None


def _fmt(score):
    return f"{score:.3f}" if score is not None else "N/A"


experiment_start = time.time()

# ── Phase A: base agent, seed = original prompt v1 ──────────────────────────
print(f"\n{'=' * 80}")
print(f"PHASE A: BASE AGENT (agent) -- seed: prompt v{SEED_PROMPT_VERSION}")
print(f"{'=' * 80}")

phase_a_remaining = [i for i in range(N_RUNS) if not _is_done("base", i)]
if phase_a_remaining:
    predict_fn_base = predict_fn_factory("agent")

for run_idx in range(N_RUNS):
    subset = subsets[run_idx]

    if _is_done("base", run_idx):
        r = _get_result("base", run_idx)
        print(f"\n--- [base] Run {run_idx + 1}/{N_RUNS} --- SKIPPED (complete: {_fmt(r['initial_score'])} -> {_fmt(r['final_score'])})")
        continue

    run_start = time.time()
    print(f"\n--- [base] Run {run_idx + 1}/{N_RUNS} ---")
    print(f"  Dataset: subset {run_idx} ({len(subset)} questions)")

    with _sp_auth():
        prompt_ref = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}/{SEED_PROMPT_VERSION}")
    obj_fn = make_objective_function(ALIGNED_JUDGE_NAME, len(subset))

    with _sp_auth():
        result = mlflow.genai.optimize_prompts(
            predict_fn=predict_fn_base,
            train_data=subset,
            prompt_uris=[prompt_ref.uri],
            optimizer=GepaPromptOptimizer(
                reflection_model=REFLECTION_MODEL,
                max_metric_calls=MAX_METRIC_CALLS,
                display_progress_bar=True,
            ),
            scorers=[aligned_judge],
            aggregation=obj_fn,
        )

    initial = getattr(result, 'initial_eval_score', None)
    final = getattr(result, 'final_eval_score', None)
    elapsed = time.time() - run_start

    results.append({
        "agent_type": "base",
        "run_idx": run_idx,
        "initial_score": initial,
        "final_score": final,
        "prompt_template": result.optimized_prompts[0].template,
        "elapsed_seconds": elapsed,
    })
    _save_checkpoint()
    print(f"  Result: {_fmt(initial)} -> {_fmt(final)} ({elapsed:.0f}s)")

total_elapsed = time.time() - experiment_start
print(f"\n{'=' * 80}")
print(f"GEPA OPTIMIZATION COMPLETE ({total_elapsed / 60:.1f} minutes)")
print(f"Total runs: {len(results)}")
print(f"{'=' * 80}")

## Step 6: Identify Best Prompt and Summarize

Find the best GEPA run and display results. The best prompt is then registered in Step 7.

In [ ]:
import pandas as pd

phase_a_results = [r for r in results if r["agent_type"] == "base"]
best_run = max(phase_a_results, key=lambda r: r["final_score"] or 0)
best_prompt_text = best_run["prompt_template"]

gepa_df = pd.DataFrame(phase_a_results)
gepa_df["lift"] = gepa_df["final_score"] / gepa_df["initial_score"]

base_init = gepa_df["initial_score"].dropna()
base_final = gepa_df["final_score"].dropna()
base_lifts = gepa_df["lift"].dropna()

print("GEPA OPTIMIZATION RESULTS")
print("=" * 90)
print(gepa_df[["run_idx", "initial_score", "final_score", "lift", "elapsed_seconds"]].to_string(index=False, float_format="%.4f"))

print(f"\n  Initial (1-5):  {base_init.mean()*5:.2f} +/- {base_init.std()*5:.2f}")
print(f"  Final (1-5):    {base_final.mean()*5:.2f} +/- {base_final.std()*5:.2f}")
print(f"  Lift:           {base_lifts.mean():.3f}x +/- {base_lifts.std():.3f}")

print(f"\n  Best run: {best_run['run_idx']} "
      f"({best_run['initial_score']:.3f} -> {best_run['final_score']:.3f})")
print(f"  Prompt hash: {hashlib.md5(best_prompt_text.encode()).hexdigest()[:12]}")

print(f"\nNext: register this prompt (@production), then run 09-Evaluation.ipynb for held-out comparison.")

## Step 7: Register Best Prompt

Register the best prompt and promote to `@production`.

In [ ]:
print(f"Best run: [base] run {best_run['run_idx']}")
print(f"  Score: {best_run['initial_score']:.3f} -> {best_run['final_score']:.3f}")
print(f"  Lift: {best_run['final_score'] / best_run['initial_score']:.2f}x")
print(f"  First 300 chars of prompt:")
print(f"  {best_prompt_text[:300]}...")

with _sp_auth():
    new_prompt = mlflow.genai.register_prompt(
        name=PROMPT_NAME,
        template=best_prompt_text,
        commit_message=(
            f"Best prompt from GEPA experiment "
            f"(run={best_run['run_idx']}, "
            f"score: {best_run['initial_score']:.3f} -> {best_run['final_score']:.3f}, "
            f"judge: {ALIGNED_JUDGE_NAME})"
        ),
        tags={
            "experiment": "gepa_optimization",
            "agent_type": "base",
            "run_idx": str(best_run["run_idx"]),
            "initial_score": str(best_run["initial_score"]),
            "final_score": str(best_run["final_score"]),
            "judge": ALIGNED_JUDGE_NAME,
        },
    )
    print(f"\nRegistered as version {new_prompt.version}")

    mlflow.genai.set_prompt_alias(
        name=PROMPT_NAME,
        alias="production",
        version=new_prompt.version,
    )
    print(f"Promoted version {new_prompt.version} to @production")

## Step 8: Save Raw Results

Display the checkpoint table. Run `09-Evaluation.ipynb` for the held-out comparison.

In [ ]:
print(f"GEPA results persisted in Delta table: {CHECKPOINT_TABLE}")
print(f"Total rows: {spark.table(CHECKPOINT_TABLE).count()}")
display(spark.table(CHECKPOINT_TABLE).select("agent_type", "run_idx", "initial_score", "final_score", "elapsed_seconds"))

print(f"\nOptimization pool persisted in: {POOL_TABLE}")
print(f"Prompt registered to @production. Run 09-Evaluation.ipynb for held-out comparison.")